In [ ]:
from pymodulon.core import IcaData
import os

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import stats
from tqdm.auto import tqdm

In [ ]:
ica_dir = "../data/ica_runs_prot/ica_runs/"

In [ ]:
X = pd.read_csv("../data/processed_data/log_normalizedCounts_norm.csv", index_col=0)

In [ ]:
def load_M(dim):
    return pd.read_csv(os.path.join(ica_dir, str(dim), "M.csv"), index_col=0)

def load_A(dim):
    return pd.read_csv(os.path.join(ica_dir, str(dim), "A.csv"), index_col=0)

In [ ]:
dims = sorted([int(x) for x in os.listdir(ica_dir) if '.' not in x])

M_data = [load_M(dim) for dim in dims]
A_data = [load_A(dim) for dim in dims]

In [ ]:
n_components = [M.shape[1] for M in M_data]

In [ ]:
# compare all dimensionalities to the highest dimensionality
final_m = M_data[-1]

#threshold for correlation comparison
thresh = 0.5

In [ ]:
m = M_data[25]
corrs = pd.DataFrame(index=final_m.columns,columns=m.columns)
for col1 in final_m.columns:
    for col2 in m.columns:
        corrs.loc[col1,col2] = abs(stats.pearsonr(final_m[col1],m[col2])[0])

In [ ]:
n_final_mods = []
for m in tqdm(M_data):
    #compute correlations of iModulons similar to highest dim
    corrs = pd.DataFrame(index=final_m.columns,columns=m.columns)
    for col1 in final_m.columns:
        for col2 in m.columns:
            corrs.loc[col1,col2] = abs(stats.pearsonr(final_m[col1],m[col2])[0])
    n_final_mods.append(len(np.where(corrs > thresh)[0]))

In [ ]:
# Find iModulons which appear to be representing the signal generated by a single gene
    # determined by the largest gene weight being twice as large as the second for a
    # given iModulon

n_single_genes = []
for m in tqdm(M_data):
    #count number of single genes
    counter = 0
    for col in m.columns:
        # sort gene weights for each iModulon
        sorted_genes = abs(m[col]).sort_values(ascending=False)
        if sorted_genes.iloc[0] > 2 * sorted_genes.iloc[1]:
            counter += 1
    n_single_genes.append(counter)

In [ ]:
# calculate the number of iModulons which are not single-gene
non_single_gene_mod = np.array(n_components) - np.array(n_single_genes)

In [ ]:
# combine statistics
DF_stats = pd.DataFrame([n_components,n_final_mods,non_single_gene_mod,n_single_genes],
                        index=['Robust Components','Final Components','Multi-gene Components',
                        'Single Gene Components'],
                        columns=dims).T
DF_stats.sort_index(inplace=True)

In [ ]:
# determine the optimal dimensionality by determining the smallest dimensionality for which 
# the number of final components (components which have high correlation with a component in the
# highest dimensionality) is greater than the number of multi-gene components for that dimensionaltiy
dimensionality = DF_stats[DF_stats['Final Components'] >= DF_stats['Multi-gene Components']].iloc[0].name
print('Optimal Dimensionality:',dimensionality)

In [ ]:
plt.plot(dims, non_single_gene_mod, label="Non-single-gene Components") 
plt.plot(dims, n_single_genes, label="Single-gene Components")
plt.plot(dims, n_components, label="Robust Components")
plt.plot(dims, n_final_mods, label="Final Components")

plt.xlabel('Dimensionality');
plt.ylabel('# Components');
plt.legend(bbox_to_anchor=(1,1));

In [ ]:
DF_stats['Explained Variance'] = 0
for M, A, dim in tqdm(zip(M_data, A_data, dims)):
    M = M.loc[X.index]
    A = A[X.columns]
    A.index = M.columns
    err_var = ((X - M.to_numpy() @ A.to_numpy())**2).sum().sum()
    orig_var = (X**2).sum().sum()
    exp_var = 1-err_var/orig_var
    DF_stats.loc[dim,'Explained Variance'] =  exp_var 

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(DF_stats)